# NB_hospital_daily_summary

Builds `<catalog>.gold.hospital_daily_summary` (one row per hospital per day) from silver lab_results, with hospital and calendar attributes and a LAG-based day-over-day test volume trend, per the Silver-to-Gold STTM. Full rebuild via CREATE OR REPLACE TABLE ... AS SELECT.

In [0]:
dbutils.widgets.text("catalog", "abd_caresync", "Catalog")
catalog = dbutils.widgets.get("catalog").strip()

silver = f"{catalog}.silver"
gold = f"{catalog}.gold"
target_table = f"{gold}.hospital_daily_summary"
print(f"Target: {target_table}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table} AS
WITH daily AS (
    SELECT
        hospital_id,
        test_date                                  AS summary_date,
        COUNT(*)                                   AS total_tests,
        COUNT(DISTINCT patient_id)                 AS unique_patients_tested,
        COUNT(DISTINCT doctor_id)                  AS active_doctor_count,
        COUNT_IF(result_status = 'Abnormal')       AS abnormal_test_count,
        COUNT_IF(result_status = 'Critical')       AS critical_test_count
    FROM {silver}.lab_results
    GROUP BY hospital_id, test_date
),
with_lag AS (
    SELECT
        d.*,
        LAG(d.total_tests) OVER (PARTITION BY d.hospital_id ORDER BY d.summary_date) AS prev_day_total_tests
    FROM daily d
)
SELECT
    CAST(w.hospital_id AS INT)                                             AS hospital_id,
    CAST(h.hospital_name AS STRING)                                        AS hospital_name,
    CAST(w.summary_date AS DATE)                                           AS summary_date,
    CAST(c.day_name AS STRING)                                             AS day_name,
    CAST(c.is_weekend AS BOOLEAN)                                          AS is_weekend,
    CAST(w.total_tests AS INT)                                             AS total_tests,
    CAST(w.unique_patients_tested AS INT)                                  AS unique_patients_tested,
    CAST(w.active_doctor_count AS INT)                                     AS active_doctor_count,
    CAST(w.abnormal_test_count AS INT)                                     AS abnormal_test_count,
    CAST(w.critical_test_count AS INT)                                     AS critical_test_count,
    CAST(ROUND(w.abnormal_test_count * 100.0 / w.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(w.critical_test_count * 100.0 / w.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct,
    CAST(w.prev_day_total_tests AS INT)                                    AS prev_day_total_tests,
    CASE
        WHEN w.prev_day_total_tests IS NULL OR w.prev_day_total_tests = 0 THEN NULL
        ELSE CAST(ROUND((w.total_tests - w.prev_day_total_tests) * 100.0 / w.prev_day_total_tests, 2) AS DECIMAL(6,2))
    END                                                                    AS test_volume_change_pct
FROM with_lag w
LEFT JOIN {silver}.hospitals h
    ON w.hospital_id = h.hospital_id
LEFT JOIN {silver}.calendar c
    ON c.date_key = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT)
""")

In [0]:
updated_df = spark.table(target_table)
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))